# cache em requests

In [4]:
import hashlib
import json
import os

from dotenv import load_dotenv
from langchain.globals import set_llm_cache
from langchain_community.cache import InMemoryCache, SQLiteCache
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv(override=True)

True

### Memória

In [5]:
openai = ChatOpenAI(model_name='gpt-4o-mini') | StrOutputParser()  # a API de completions (gpt-3.5-turbo-instruct) foi desativada
set_llm_cache(InMemoryCache()) # única configuração necessária para usar o cache em memoria

In [6]:
prompt = 'Me diga em poucas palavras oque é DOTA2.'
response1 = openai.invoke(prompt)
print("Primeira resposta (API chamada):", response1)

Primeira resposta (API chamada): Dota 2 é um jogo online de estratégia em equipe, do gênero multiplayer online battle arena (MOBA), desenvolvido pela Valve Corporation. Nele, duas equipes de cinco jogadores competem para destruir a base inimiga, utilizando heróis com habilidades únicas. É conhecido por sua complexidade e profundidade estratégica.


In [4]:
response2 = openai.invoke(prompt)
print("Segunda resposta (usando cache):", response2)

Segunda resposta (usando cache): 

DOTA2 é um jogo eletrônico de estratégia em tempo real, no estilo MOBA, em que duas equipes de cinco jogadores batalham pelo controle de um mapa e tentam destruir a base inimiga. Cada jogador controla um herói com habilidades únicas e deve trabalhar em equipe para vencer.


### Disco / Banco de dados

In [ ]:
set_llm_cache(SQLiteCache(database_path="openai_cache.db")) # única configuração necessária para usar o cache disco

In [ ]:
prompt = 'Me diga em poucas palavras quem foi Tim Maia.'

In [7]:
response1 = openai.invoke(prompt)
print("Primeira resposta (API chamada):", response1)

Primeira resposta (API chamada): 

Tim Maia foi um cantor, compositor e produtor musical brasileiro, considerado um dos maiores ícones da música popular brasileira. Ele teve uma carreira marcada por sucessos, polêmicas e influências do soul, funk e outros gêneros musicais. Sua voz marcante e estilo único o tornaram um ícone da cultura brasileira.


In [8]:
response2 = openai.invoke(prompt)
print("Segunda resposta (usando cache):", response2)

Segunda resposta (usando cache): 

Tim Maia foi um cantor, compositor e produtor musical brasileiro, considerado um dos maiores ícones da música popular brasileira. Ele teve uma carreira marcada por sucessos, polêmicas e influências do soul, funk e outros gêneros musicais. Sua voz marcante e estilo único o tornaram um ícone da cultura brasileira.


### Personalizado

In [ ]:
class SimpleDiskCache:
    def __init__(self, cache_dir='cache_dir'):
        self.cache_dir = cache_dir
        os.makedirs(self.cache_dir, exist_ok=True)

    def _get_cache_path(self, key):
        hashed_key = hashlib.md5(key.encode()).hexdigest() #hasg cria nome de arquivo único
        return os.path.join(self.cache_dir, f"{hashed_key}.json")

    def lookup(self, key, llm_string):
        cache_path = self._get_cache_path(key)
        if os.path.exists(cache_path):
            with open(cache_path, 'r') as f:
                return json.load(f)
        return None

    def update(self, key, value, llm_string):
        cache_path = self._get_cache_path(key)
        with open(cache_path, 'w') as f:
            json.dump(value, f)

In [ ]:
cache = SimpleDiskCache()
set_llm_cache(cache)
prompt = 'Me diga em poucas palavras quem é The white buffalo.'

def invoke_with_cache(llm, prompt, cache):
    cached_response = cache.lookup(prompt, "")
    if cached_response:
        print("Usando cache:")
        return cached_response

    response = llm.invoke(prompt)
    cache.update(prompt, response, "")
    return response

In [ ]:
response1 = invoke_with_cache(openai, prompt, cache)
response_text1 = response1.replace('\n', ' ') 

print("Primeira resposta (API chamada):", response_text1)

Primeira resposta (API chamada):   The White Buffalo é um cantor, compositor e músico americano conhecido por sua voz potente e suas letras introspectivas, que misturam influências do blues, folk e country. Seu estilo musical é descrito como uma mistura única de rock americano e música folk tradicional. Ele ganhou destaque por suas músicas serem usadas em séries de TV e filmes, e é reconhecido por suas apresentações ao vivo emocionantes.


In [ ]:
response2 = invoke_with_cache(openai, prompt, cache)
response_text2 = response2.replace('\n', ' ')  

print("Segunda resposta (usando cache):", response_text2)

Usando cache:
Segunda resposta (usando cache):   The White Buffalo é um cantor, compositor e músico americano conhecido por sua voz potente e suas letras introspectivas, que misturam influências do blues, folk e country. Seu estilo musical é descrito como uma mistura única de rock americano e música folk tradicional. Ele ganhou destaque por suas músicas serem usadas em séries de TV e filmes, e é reconhecido por suas apresentações ao vivo emocionantes.
